# Get the key and values for the dataset

Remember: max 5 queries per minute!


In [1]:
import pandas as pd
import time
import sdmx 

client = sdmx.Client("ISTAT")

In [2]:
# get all available dataflows
flow_msg = client.dataflow()
dataflows = sdmx.to_pandas(flow_msg.dataflow)
#dataflows.to_csv("../data/all_istat_dataflows.csv")
dataflows

101_1015                                                                        Crops
101_1015_DF_DCSP_COLTIVAZIONI_1                   Areas and production - overall data
101_1015_DF_DCSP_COLTIVAZIONI_10                                      Sowing forecast
101_1015_DF_DCSP_COLTIVAZIONI_2       Areas and production - overall data - provinces
101_1030                                            PDO, PGI and TSG quality products
                                                          ...                        
SDDS_PLUS_NAG_08_DF_163_156_2       National accounts (GDP) - chained volume in mo...
SDDS_PLUS_POP_DF_22_315_1                                                  Population
TEST_LINKED_DF_DCSP_PREZZIAGR_1                       TEST_LINKED_DF_DCSP_PREZZIAGR_1
UEM                                                                      Unemployment
WOE                                                                 Contractual wages
Length: 4836, dtype: str

In [ ]:
ds_id = '26_295_DF_DCIS_MORTALITA1_2' # Change it to explore other datasets
ds_msg = client.dataflow(ds_id)
ds_flow = list(ds_msg.dataflow.values())[0]
dsd = ds_flow.structure
print(f"Dataset `{ds_id}` has the following dimensions: {dsd.dimensions.components}")

Dataset `26_295_DF_DCIS_MORTALITA1_2` has the following dimensions: [<Dimension FREQ>, <Dimension REF_AREA>, <Dimension DATA_TYPE>, <Dimension SEX>, <Dimension AGE>, <TimeDimension TIME_PERIOD>]


In [20]:
ds_components = {
    "164_279_DF_DCIS_RICPOPRES1991_1": [],
    "164_305_DF_DCIS_RICPOPRES2001_1": [],
    "164_164_DF_DCIS_RICPOPRES2011_1": [],
    "22_289_DF_DCIS_POPRES1_1": []
}
for ds_id in ds_components.keys():
    ds_msg = client.dataflow(ds_id)
    ds_flow = list(ds_msg.dataflow.values())[0]
    dsd = ds_flow.structure
    print(f"Dataset `{ds_id}` has the following dimensions: {dsd.dimensions.components}")
    ds_components[ds_id] = list(dsd.dimensions.components) 

Dataset `164_279_DF_DCIS_RICPOPRES1991_1` has the following dimensions: [<Dimension FREQ>, <Dimension REF_AREA>, <Dimension DATA_TYPE>, <Dimension AGE>, <Dimension SEX>, <TimeDimension TIME_PERIOD>]
Dataset `164_305_DF_DCIS_RICPOPRES2001_1` has the following dimensions: [<Dimension FREQ>, <Dimension REF_AREA>, <Dimension DATA_TYPE>, <Dimension AGE>, <Dimension SEX>, <TimeDimension TIME_PERIOD>]
Dataset `164_164_DF_DCIS_RICPOPRES2011_1` has the following dimensions: [<Dimension FREQ>, <Dimension REF_AREA>, <Dimension DATA_TYPE>, <Dimension AGE>, <Dimension SEX>, <Dimension CITIZENSHIP>, <TimeDimension TIME_PERIOD>]
Dataset `22_289_DF_DCIS_POPRES1_1` has the following dimensions: [<Dimension FREQ>, <Dimension REF_AREA>, <Dimension DATA_TYPE>, <Dimension SEX>, <Dimension AGE>, <Dimension MARITAL_STATUS>, <TimeDimension TIME_PERIOD>]


In [13]:
unique_components = {comp for components in ds_components.values() for comp in components}

parameter_to_name = {}
for dim in unique_components:
    print(f"Dimension: {dim}")
    if dim.local_representation.enumerated is not None:
        lc_df = sdmx.to_pandas(dim.local_representation.enumerated)
        if isinstance(lc_df, pd.Series):
            lc_df = lc_df.to_frame()
        display(lc_df)
        parameter_to_name[dim.id] = lc_df.iloc[:,0].to_dict()
        #time.sleep(15) # no need to sleep for this

Dimension: SEX


,Gender
CL_SEXISTAT1,
1,males
2,females
3,n.a.
9,total
M,males
F,females
NRP,no responce
T,total


Dimension: TIME_PERIOD
Dimension: MARITAL_STATUS


,Marital status
CL_STATCIV2,
1,never married
2,married
11,de facto separated
12,legally separated
10,married or legally separated
3,divorced
4,widowed
5,divorced or married
6,divorced or widowed


Dimension: REF_AREA


...

,Citizenship
CL_CITTADINANZA,
ITL,italian
FRG,foreign
EU27,of European Union country with 27 Member States
EU27_NOITL,of European Union country (except Italy)
EXTEU27,of extra European Union country with 27 Member...
APO,stateless
FRGAPO,foreign/stateless
TOTAL,total


Dimension: DATA_TYPE


,Data type
CL_TIPO_DATO15,
POPSTAT,statistical population on 1st January
JAN,population on 1st January
JAN_C,population on 1st January within the borders o...
BEG,population censused on 1st January
LBIRTH,live births
...,...
IMUNOTHREG,immigrated from municipalities of other regions
IMUNSAMREG,immigrated from municipalities in the same region
EMUNOTHREG,emigrated to municipalities in other regions


Dimension: AGE


,Age class
CL_ETA1,
NSP,no response
D0,0 days
D1,1 day
D2,2 days
D3,3 days
...,...
Y9-12,9-12 years
Y5-8,5-8 years
Y_UN8,until 8 years


In [3]:
df1971 = sdmx.to_pandas(
    client.data(
        resource_id="164_346_DF_DCIS_RICPOPRES1971_1", 
        key={
            "FREQ": "A",
            "REF_AREA": "IT",
            "DATA_TYPE": "JAN",     
            "AGE": [], # I want them all
            "SEX": "9",
            #"CITIZENSHIP": "TOTAL", # not present in this dataset   
        })
    ).reset_index()
df1971

,FREQ,REF_AREA,DATA_TYPE,AGE,SEX,TIME_PERIOD,value
0,A,IT,JAN,TOTAL,9,1952,47538945.0
1,A,IT,JAN,TOTAL,9,1953,47792111.0
2,A,IT,JAN,TOTAL,9,1954,48122579.0
3,A,IT,JAN,TOTAL,9,1955,48476569.0
4,A,IT,JAN,TOTAL,9,1956,48788543.0
...,...,...,...,...,...,...,...
2035,A,IT,JAN,Y99,9,1967,259.0
2036,A,IT,JAN,Y99,9,1968,475.0
2037,A,IT,JAN,Y99,9,1969,475.0
2038,A,IT,JAN,Y99,9,1970,1043.0


In [4]:
df1981 = sdmx.to_pandas(
    client.data(
        resource_id="164_347_DF_DCIS_RICPOPRES1981_1", 
        key={
            "FREQ": "A",
            "REF_AREA": "IT",
            "DATA_TYPE": "JAN",     
            "AGE": [], # I want them all
            "SEX": "9",
            #"CITIZENSHIP": "TOTAL", # not present in this dataset   
        })
    ).reset_index()
df1981

,FREQ,REF_AREA,DATA_TYPE,AGE,SEX,TIME_PERIOD,value
0,A,IT,JAN,TOTAL,9,1972,54188579.0
1,A,IT,JAN,TOTAL,9,1973,54574111.0
2,A,IT,JAN,TOTAL,9,1974,54928700.0
3,A,IT,JAN,TOTAL,9,1975,55293036.0
4,A,IT,JAN,TOTAL,9,1976,55588966.0
...,...,...,...,...,...,...,...
1015,A,IT,JAN,Y99,9,1977,700.0
1016,A,IT,JAN,Y99,9,1978,807.0
1017,A,IT,JAN,Y99,9,1979,866.0
1018,A,IT,JAN,Y99,9,1980,849.0


In [15]:
df0 = sdmx.to_pandas(
    client.data(
        resource_id="164_279_DF_DCIS_RICPOPRES1991_1", 
        key={
            "FREQ": "A",
            "REF_AREA": "IT",
            "DATA_TYPE": "JAN",     
            "AGE": [], # I want them all
            "SEX": "9",
            #"CITIZENSHIP": "TOTAL", # not present in this dataset   
        })
    ).reset_index()
df0

,FREQ,REF_AREA,DATA_TYPE,AGE,SEX,TIME_PERIOD,value
0,A,IT,JAN,TOTAL,9,1982,56524064.0
1,A,IT,JAN,TOTAL,9,1983,56563031.0
2,A,IT,JAN,TOTAL,9,1984,56565117.0
3,A,IT,JAN,TOTAL,9,1985,56588319.0
4,A,IT,JAN,TOTAL,9,1986,56597823.0
5,A,IT,JAN,TOTAL,9,1987,56594487.0
6,A,IT,JAN,TOTAL,9,1988,56609375.0
7,A,IT,JAN,TOTAL,9,1989,56649201.0
8,A,IT,JAN,TOTAL,9,1990,56694360.0
9,A,IT,JAN,TOTAL,9,1991,56744119.0


In [21]:
df00 = sdmx.to_pandas(
    client.data(
        resource_id="164_305_DF_DCIS_RICPOPRES2001_1", 
        key={
            "FREQ": "A",
            "REF_AREA": "IT",
            "DATA_TYPE": "JAN",     
            "AGE": [], # I want them all
            "SEX": "9",
            #"CITIZENSHIP": "TOTAL", # not present in this dataset   
        })
    ).reset_index()
df00

,FREQ,REF_AREA,DATA_TYPE,AGE,SEX,TIME_PERIOD,value
0,A,IT,JAN,TOTAL,9,1992,56772923.0
1,A,IT,JAN,TOTAL,9,1993,56821250.0
2,A,IT,JAN,TOTAL,9,1994,56842392.0
3,A,IT,JAN,TOTAL,9,1995,56844408.0
4,A,IT,JAN,TOTAL,9,1996,56844197.0
...,...,...,...,...,...,...,...
1015,A,IT,JAN,Y99,9,1997,3049.0
1016,A,IT,JAN,Y99,9,1998,2940.0
1017,A,IT,JAN,Y99,9,1999,3167.0
1018,A,IT,JAN,Y99,9,2000,3910.0


In [16]:
df1 = sdmx.to_pandas(
    client.data(
        resource_id="164_164_DF_DCIS_RICPOPRES2011_1", 
        key={
            "FREQ": "A",
            "REF_AREA": "IT",
            "DATA_TYPE": "JAN",     
            "AGE": [], # I want them all
            "SEX": "9",
            "CITIZENSHIP": "TOTAL",    
        })
    ).reset_index()
df1

,FREQ,REF_AREA,DATA_TYPE,AGE,SEX,CITIZENSHIP,TIME_PERIOD,value
0,A,IT,JAN,TOTAL,9,TOTAL,2001,56995744.0
1,A,IT,JAN,TOTAL,9,TOTAL,2002,56993270.0
2,A,IT,JAN,TOTAL,9,TOTAL,2003,57186378.0
3,A,IT,JAN,TOTAL,9,TOTAL,2004,57611990.0
4,A,IT,JAN,TOTAL,9,TOTAL,2005,58044368.0
...,...,...,...,...,...,...,...,...
1832,A,IT,JAN,Y99,9,TOTAL,2015,10728.0
1833,A,IT,JAN,Y99,9,TOTAL,2016,8258.0
1834,A,IT,JAN,Y99,9,TOTAL,2017,7145.0
1835,A,IT,JAN,Y99,9,TOTAL,2018,6689.0


In [ ]:
df2 = sdmx.to_pandas(
    client.data(
        resource_id="22_289_DF_DCIS_POPRES1_1", 
        key={
            "FREQ": "A",
            "REF_AREA": "IT",
            "DATA_TYPE": "JAN",     
            "AGE": [], # I want them all
            "SEX": "9",
            "MARITAL_STATUS": "99",
            #"CITIZENSHIP": "TOTAL", # not present in this dataset
        })
    ).reset_index()
df2

,FREQ,REF_AREA,DATA_TYPE,SEX,AGE,MARITAL_STATUS,TIME_PERIOD,value
0,A,IT,JAN,9,TOTAL,99,2019,59816673.0
1,A,IT,JAN,9,TOTAL,99,2020,59641488.0
2,A,IT,JAN,9,TOTAL,99,2021,59236213.0
3,A,IT,JAN,9,TOTAL,99,2022,59030133.0
4,A,IT,JAN,9,TOTAL,99,2023,58997201.0
...,...,...,...,...,...,...,...,...
811,A,IT,JAN,9,Y99,99,2022,13742.0
812,A,IT,JAN,9,Y99,99,2023,13403.0
813,A,IT,JAN,9,Y99,99,2024,13839.0
814,A,IT,JAN,9,Y99,99,2025,13994.0
